# Sorting dead ends from actual branching points

Input:
- Skeleton_csv file
- Image dimensions in units of voxel

Output:
- csv table including filename, number of dead ends, number of border points and number of actual branching points
- Images with spheres at the respective coordinates

## Load data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from skimage import io

In [2]:
# Input: Image dimensions
image_width = 1012
image_height = 590
image_depth = 542

In [3]:
# Input: Pixel dimensions (µm per pixel)
pixel_size_x = 2.6 # pixel size in X (µm)
pixel_size_y = 2.6 # pixel size in Y (µm)
pixel_size_z = 5.0 # pixel size in Z (µm)

In [4]:
# Filename
filename = "VK-AA499_shaft_590-1180_"

# Parameters
min_length = 25

In [5]:
# Load CSV file
csv_file = filename + 'skeleton.csv'
data = pd.read_csv(csv_file)
print(len(data))

# Convert coordinates from µm to pixels and round
for coord in ['V1 x', 'V2 x']:
    data[coord] = (data[coord] / pixel_size_x).round().astype(int)
for coord in ['V1 y', 'V2 y']:
    data[coord] = (data[coord] / pixel_size_y).round().astype(int)
for coord in ['V1 z', 'V2 z']:
        data[coord] = (data[coord] / pixel_size_z).round().astype(int)

11595


In [6]:
# 1. Remove branches shorter than 25 µm
filtered_data = data[data['Branch length'] >= min_length]
print(len(filtered_data))

# 2. Remove border points (y = 0 or y = max(y))
filtered_data2 = filtered_data[(filtered_data['V1 y'] > 0) & (filtered_data['V2 y'] > 0)]
filtered_data2 = filtered_data[(filtered_data['V1 y'] < image_height - 1) & (filtered_data['V2 y'] < image_height -1)]

print(len(filtered_data2))

9234
9214


In [7]:
# 3. Identify dead-end points (unique endpoints)
endpoints = pd.concat([
    filtered_data2[['V1 x', 'V1 y', 'V1 z']].rename(columns={'V1 x': 'x', 'V1 y': 'y', 'V1 z': 'z'}),
    filtered_data2[['V2 x', 'V2 y', 'V2 z']].rename(columns={'V2 x': 'x', 'V2 y': 'y', 'V2 z': 'z'})
])

endpoints_counts = endpoints.value_counts().reset_index(name='count')
dead_ends = endpoints_counts[endpoints_counts['count'] == 1]

In [8]:
# 4. Summarize frequencies
border_points = len(filtered_data) - len(filtered_data2)
death_end_count = len(dead_ends)
branching_points = len(endpoints_counts[endpoints_counts['count'] > 2])

print(len(filtered_data), border_points, death_end_count, branching_points)

9234 20 1357 3918


In [9]:
# 5. Create and save visualization images
sphere_radius = 2

# Helper function to generate sphere masks
def generate_sphere(image_shape, points, radius):
    img = np.zeros(image_shape, dtype=np.uint8)
    for _, point in points.iterrows():
        x, y, z = int(point[0]), int(point[1]), int(point[2])
        for i in range(-radius, radius + 1):
            for j in range(-radius, radius + 1):
                for k in range(-radius, radius + 1):
                    if 0 <= x + i < image_shape[0] and 0 <= y + j < image_shape[1] and 0 <= z + k < image_shape[2]:
                        if i**2 + j**2 + k**2 <= radius**2:
                            img[x + i, y + j, z + k] = 255
    return img

# Generate images
image_shape = (image_width, image_height, image_depth)

dead_end_img = generate_sphere(image_shape, dead_ends, sphere_radius)
branching_img = generate_sphere(image_shape, endpoints_counts[endpoints_counts['count'] > 2], sphere_radius)

# Save the images
io.imsave(filename + 'DE.tif', dead_end_img)
io.imsave(filename + 'BP.tif', branching_img)

print("Images saved")

/tmp/ipykernel_1111264/1942662996.py:24: UserWarning: VK-AA499_shaft_590-1180_DE.tif is a low contrast image
  io.imsave(filename + 'DE.tif', dead_end_img)
/tmp/ipykernel_1111264/1942662996.py:25: UserWarning: VK-AA499_shaft_590-1180_BP.tif is a low contrast image
  io.imsave(filename + 'BP.tif', branching_img)


Images saved
